In [1]:
import pandas as pd
import numpy as np

cc = pd.read_csv("cookie_cats.csv")

print(cc.isnull().sum())
print("Duplicate users:", cc["userid"].duplicated().sum())
print(cc["version"].value_counts())
print(cc.groupby("version")[["retention_1", "retention_7"]].mean())

userid            0
version           0
sum_gamerounds    0
retention_1       0
retention_7       0
dtype: int64
Duplicate users: 0
version
gate_40    45489
gate_30    44700
Name: count, dtype: int64
         retention_1  retention_7
version                          
gate_30     0.448188     0.190201
gate_40     0.442283     0.182000


In [2]:
from statsmodels.stats.proportion import proportions_ztest, confint_proportions_2indep

def ab_test(df, metric):
    a = df[df["version"] == "gate_30"][metric]
    b = df[df["version"] == "gate_40"][metric]

    counts = np.array([b.sum(), a.sum()])
    nobs = np.array([len(b), len(a)])

    z, p = proportions_ztest(counts, nobs)
    lo, hi = confint_proportions_2indep(counts[0], nobs[0], counts[1], nobs[1], compare="diff")

    pa, pb = a.mean(), b.mean()
    return {
        "metric": metric,
        "gate_30 (control)": round(pa * 100, 2),
        "gate_40 (treatment)": round(pb * 100, 2),
        "abs_diff_pp": round((pb - pa) * 100, 2),
        "relative_lift_%": round((pb - pa) / pa * 100, 2),
        "p_value": round(p, 4),
        "95%_CI_diff_pp": (round(lo * 100, 2), round(hi * 100, 2)),
    }

pd.DataFrame([ab_test(cc, "retention_1"), ab_test(cc, "retention_7")]).T

,0,1
metric,retention_1,retention_7
gate_30 (control),44.82,19.02
gate_40 (treatment),44.23,18.2
abs_diff_pp,-0.59,-0.82
relative_lift_%,-1.32,-4.31
p_value,0.0744,0.0016
95%_CI_diff_pp,"(-1.24, 0.06)","(-1.33, -0.31)"


In [3]:
from scipy.stats import chisquare

obs = cc["version"].value_counts().reindex(["gate_30", "gate_40"]).values
exp = [obs.sum() / 2] * 2
chi, p = chisquare(obs, exp)
print("Observed:", obs, "| chi2:", round(chi, 2), "| p-value:", round(p, 4))

Observed: [44700 45489] | chi2: 6.9 | p-value: 0.0086


In [4]:
print(cc["sum_gamerounds"].describe())
print(cc.sort_values("sum_gamerounds", ascending=False).head(5))
print("Users with 0 rounds:", (cc["sum_gamerounds"] == 0).sum())

count    90189.000000
mean        51.872457
std        195.050858
min          0.000000
25%          5.000000
50%         16.000000
75%         51.000000
max      49854.000000
Name: sum_gamerounds, dtype: float64
        userid  version  sum_gamerounds  retention_1  retention_7
57702  6390605  gate_30           49854        False         True
7912    871500  gate_30            2961         True         True
29417  3271615  gate_40            2640         True        False
43671  4832608  gate_30            2438         True         True
48188  5346171  gate_40            2294         True         True
Users with 0 rounds: 3994


In [5]:
rng = np.random.default_rng(42)
a = cc[cc["version"] == "gate_30"]["retention_7"].values
b = cc[cc["version"] == "gate_40"]["retention_7"].values

diffs = [
    rng.choice(b, len(b)).mean() - rng.choice(a, len(a)).mean()
    for _ in range(5000)
]
lo, hi = np.percentile(diffs, [2.5, 97.5])
print(f"Bootstrap 95% CI (pp): ({lo*100:.2f}, {hi*100:.2f})")
print("P(gate_40 better):", round((np.array(diffs) > 0).mean(), 4))

Bootstrap 95% CI (pp): (-1.33, -0.32)
P(gate_40 better): 0.001


In [6]:
from scipy.stats import mannwhitneyu

# 1) Engagement (outlier-safe): median + Mann-Whitney
print(cc.groupby("version")["sum_gamerounds"].median())
a = cc[cc["version"] == "gate_30"]["sum_gamerounds"]
b = cc[cc["version"] == "gate_40"]["sum_gamerounds"]
print("Mann-Whitney p-value:", round(mannwhitneyu(a, b).pvalue, 4))

# 2) Day-7 test dobara, 0-round users aur outlier hata ke
clean = cc[(cc["sum_gamerounds"] > 0) & (cc["sum_gamerounds"] < 10000)]
print(len(cc) - len(clean), "rows removed")
pd.DataFrame([ab_test(clean, "retention_7")]).T

version
gate_30    17.0
gate_40    16.0
Name: sum_gamerounds, dtype: float64
Mann-Whitney p-value: 0.0502
3995 rows removed


,0
metric,retention_7
gate_30 (control),19.84
gate_40 (treatment),19.03
abs_diff_pp,-0.81
relative_lift_%,-4.08
p_value,0.0026
95%_CI_diff_pp,"(-1.34, -0.28)"
